In [ ]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

spark = SparkSession.builder \
    .appName("04_Gold_KPIs") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark OK :", spark.version)

In [ ]:
SILVER = "C:/Users/smagu/retail-data-platform/data/silver"

# Lecture des tables Silver via pandas puis conversion Spark
# (evite winutils sur Windows)
clients      = spark.createDataFrame(pd.read_parquet(f"{SILVER}/clients/clients.parquet"))
employes     = spark.createDataFrame(pd.read_parquet(f"{SILVER}/employes/employes.parquet"))
fournisseurs = spark.createDataFrame(pd.read_parquet(f"{SILVER}/fournisseurs/fournisseurs.parquet"))
produits     = spark.createDataFrame(pd.read_parquet(f"{SILVER}/produits/produits.parquet"))
ventes       = spark.createDataFrame(pd.read_parquet(f"{SILVER}/ventes/ventes.parquet"))

print("Tables Silver chargees :")
for nom, df in [("clients", clients), ("employes", employes), ("fournisseurs", fournisseurs),
                ("produits", produits), ("ventes", ventes)]:
    print(f"  {nom} : {df.count()} lignes")

In [ ]:
# KPI 1 : Chiffre d'affaires par annee et par mois
ca_par_mois = ventes \
    .groupBy("Annee", "Mois") \
    .agg(F.round(F.sum("MontantTotal"), 2).alias("CA_Total"),
         F.count("VenteID").alias("Nb_Ventes")) \
    .orderBy("Annee", "Mois")

print("=== CA PAR ANNEE ET MOIS ===")
ca_par_mois.show(24, truncate=False)

In [ ]:
# KPI 2 : Top 10 produits par chiffre d'affaires
top_produits = ventes \
    .join(produits, on="ProduitID", how="left") \
    .groupBy("ProduitID", "NomProduit") \
    .agg(F.round(F.sum("MontantTotal"), 2).alias("CA_Total"),
         F.sum("QuantiteVendue").alias("Qte_Vendue"),
         F.count("VenteID").alias("Nb_Ventes")) \
    .orderBy(F.desc("CA_Total")) \
    .limit(10)

print("=== TOP 10 PRODUITS PAR CA ===")
top_produits.show(truncate=False)

In [ ]:
# KPI 3 : Top 10 clients par chiffre d'affaires
top_clients = ventes \
    .join(clients, on="ClientID", how="left") \
    .groupBy("ClientID", "Nom", "Prenom") \
    .agg(F.round(F.sum("MontantTotal"), 2).alias("CA_Total"),
         F.count("VenteID").alias("Nb_Achats")) \
    .orderBy(F.desc("CA_Total")) \
    .limit(10)

print("=== TOP 10 CLIENTS PAR CA ===")
top_clients.show(truncate=False)

In [ ]:
# KPI 4 : Performance des employes
perf_employes = ventes \
    .join(employes, on="EmployeID", how="left") \
    .groupBy("EmployeID", "Nom", "Prenom", "Fonction") \
    .agg(F.round(F.sum("MontantTotal"), 2).alias("CA_Genere"),
         F.count("VenteID").alias("Nb_Ventes"),
         F.round(F.avg("MontantTotal"), 2).alias("Panier_Moyen")) \
    .orderBy(F.desc("CA_Genere"))

print("=== PERFORMANCE DES EMPLOYES ===")
perf_employes.show(10, truncate=False)

In [ ]:
# KPI 5 : CA par fournisseur
ca_fournisseurs = ventes \
    .join(produits, on="ProduitID", how="left") \
    .join(fournisseurs, on="FournisseurID", how="left") \
    .groupBy("FournisseurID", "NomFournisseur") \
    .agg(F.round(F.sum("MontantTotal"), 2).alias("CA_Total"),
         F.count("VenteID").alias("Nb_Ventes"),
         F.countDistinct("ProduitID").alias("Nb_Produits")) \
    .orderBy(F.desc("CA_Total")) \
    .limit(10)

print("=== TOP 10 FOURNISSEURS PAR CA ===")
ca_fournisseurs.show(truncate=False)

In [ ]:
# Ecriture Gold en Parquet (via pandas/pyarrow, sans winutils)
GOLD = "C:/Users/smagu/retail-data-platform/data/gold"

gold_tables = {
    "ca_par_mois":     ca_par_mois,
    "top_produits":    top_produits,
    "top_clients":     top_clients,
    "perf_employes":   perf_employes,
    "ca_fournisseurs": ca_fournisseurs,
}

for nom, df in gold_tables.items():
    path = f"{GOLD}/{nom}"
    os.makedirs(path, exist_ok=True)
    df.toPandas().to_parquet(f"{path}/{nom}.parquet", index=False)
    print(f"Ecrit : {path}/{nom}.parquet")

print("\nToutes les tables Gold sont enregistrees !")

In [ ]:
# Verification finale
print("=== VERIFICATION GOLD ===")
for nom in gold_tables.keys():
    path = f"{GOLD}/{nom}/{nom}.parquet"
    df_check = pd.read_parquet(path)
    print(f"\n{nom.upper()} — {len(df_check)} lignes, colonnes : {list(df_check.columns)}")

print("\nNotebook 04 termine !")